## CNN VS FINE-TUNING VS RANDOM FOREST


Implementando Pytorch/Torchvision, podemos comparar o desempenño de um modelo CNN pré-treinado, un modelo fine-tuning y unmodelo Random Forest para la tarea de clasificacion de imagenes.

El imput para probar nuestros modelos sera un archivo de audio nuevo y grabado por nosotros mismos, este clip primero sera filtrado, recortado y convertido a un espectrograma, los parametros de dicha transformación deberan ajustarse al formato de entrada al que hemos configurado como nuevos inputs para nuestros modelos.

Para ello sera destinada una web app, donde el usuario pueda subir su clip de audio, y el sistema se encargara de procesarlo y mostrar los resultados de las predicciones de cada modelo.

In [ ]:
# importando librerias necesarias

import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from torch.cuda.amp import GradScaler, autocast # For memory-efficient training
from google.colab import drive 


In [ ]:
# Mount Google Drive to access the dataset

drive.mount('content/drive')
ROOT_DIR = '/content/drive/MyDrive/ravdess_images'  # Cambia esto a tu ruta

# Configuración del dispositivo (GPU si está disponible, de lo contrario CPU)
# Se recomienda usar la GPU que ofrece Google Colab para acelerar el entrenamiento de

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Estas en modo: {device}")

class RavdessImageDataset(Dataset):
    def __init__(self, root_dir, feature_type='melspect', transform=None):
        """
        root_dir: La carpeta raíz que contiene las subcarpetas de emociones.
        feature_type: El feature especifico dentro de la carpeta (e.g., 'melspect', 'MFCC').
        """
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        # Obtener la lista de emociones (subcarpetas) y asignarles un índice
        self.emotions = sorted(os.listdir(root_dir))
        self.class_to_idx = {emotion: idx for idx, emotion in enumerate(self.emotions)}
        
        for emotion in self.emotions:
            # Construir la ruta a la carpeta de características específica para esta emoción
            feature_dir = os.path.join(root_dir, emotion, feature_type)
            if os.path.isdir(feature_dir):
                # Iterar sobre las imágenes en esta carpeta en png, jpg o jpeg
                for img_path in glob.glob(os.path.join(feature_dir, '*.*')):
                    if img_path.endswith(('.png', '.jpg', '.jpeg')):
                        self.image_paths.append(img_path)
                        self.labels.append(self.class_to_idx[emotion])
                        
    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        # Convertir a RGB para asegurarnos de que todas las imágenes tengan 3 canales (CNN espera 3 canales)
        image = Image.open(img_path).convert('RGB') 
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

# Preprocesando Para Fine-Tunning: Resize para ResNet (128x128), convertir a tensor y, normalizar usando ImageNet stats
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])